# Stage 2 — SEA-LION Annotation Validation & Train/Val Split

**Steps:**
1. Compute inter-annotator agreement on Pool A (Spearman ρ + Cohen's Kappa)
2. Gate: if Spearman ρ > 0.75, SEA-LION labels are trusted as training signal
3. Split 5,000-image SEA-LION labels → train (4,000) / val (1,000) for ViT fine-tuning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from pathlib import Path

SCORING_DIR = Path("..").resolve()   # scoring/ root
STAGE2_DIR  = Path(".").resolve()    # stage2_validation_and_split/
sns.set_theme(style="whitegrid", palette="muted")

## 0. LLM Judge Selection — GPT-4o vs Qwen3-VL-8B vs SEA-LION

Before committing to 5,000-image labelling, we compare three candidate VLM judges on a shared **100-image validation set** (50 bg60k + 50 ecommerce118k) to understand their scoring behaviour and discriminative range.

The key criterion: a model whose scores cluster in a narrow high range provides poor regression training signal. We want a model that uses the full 1–10 scale and is sensitive to real quality differences — especially for the harder ecommerce118k images.

In [ ]:
JUDGE_DIR = SCORING_DIR / "stage2_judge_comparison"

judges = {
    "GPT-4o":       pd.read_csv(JUDGE_DIR / "vlm_scores_gpt4o_100img.csv"),
    "Qwen3-VL-8B":  pd.read_csv(JUDGE_DIR / "vlm_scores_qwen3vl8b_100img.csv"),
    "SEA-LION":     pd.read_csv(JUDGE_DIR / "vlm_scores_sealion_100img.csv"),
}

MODEL_COLORS = {"GPT-4o": "#27ae60", "Qwen3-VL-8B": "#2980b9", "SEA-LION": "#e67e22"}

# Merge on (dataset, filename)
merged_judges = judges["GPT-4o"][["dataset", "filename", "category"]].copy()
for name, df in judges.items():
    merged_judges = merged_judges.merge(
        df[["dataset", "filename", "score"]].rename(columns={"score": f"score_{name}"}),
        on=["dataset", "filename"], how="inner"
    )

score_cols_j = [f"score_{m}" for m in judges]
print(f"Validation set: {len(merged_judges)} images ({merged_judges['dataset'].value_counts().to_dict()})")
print()

# Distribution stats
rows = []
for name in judges:
    col = f"score_{name}"
    for ds in ["All", "bg60k", "ecommerce118k"]:
        s = merged_judges[col] if ds == "All" else merged_judges.loc[merged_judges["dataset"] == ds, col]
        rows.append({
            "Model": name, "Dataset": ds,
            "Mean": s.mean().round(2), "Median": s.median().round(2),
            "Std": s.std().round(2), "IQR": (s.quantile(0.75) - s.quantile(0.25)).round(2),
            "Min": s.min(), "Max": s.max(),
        })

stats_jdf = pd.DataFrame(rows).set_index(["Model", "Dataset"])
display(stats_jdf)

In [ ]:
# Score distribution visualisation — highlights ceiling effects in GPT-4o & Qwen3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: KDE per model (all images)
ax = axes[0]
for name, col in zip(judges, score_cols_j):
    merged_judges[col].plot.kde(ax=ax, color=MODEL_COLORS[name], lw=2.2, label=name)
    ax.axvline(merged_judges[col].mean(), color=MODEL_COLORS[name], lw=1, ls="--", alpha=0.6)
ax.set_xlabel("Score")
ax.set_ylabel("Density")
ax.set_xlim(1, 11)
ax.set_title("Score Density (all 100 images)", fontsize=12, fontweight="bold")
ax.legend()

# Right: Box plots per model × dataset
long_j = merged_judges.melt(
    id_vars=["dataset"], value_vars=score_cols_j,
    var_name="model", value_name="score"
)
long_j["model"] = long_j["model"].str.replace("score_", "", regex=False)
sns.boxplot(data=long_j, x="model", y="score", hue="dataset",
            palette={"bg60k": "#8e44ad", "ecommerce118k": "#c0392b"},
            order=list(judges.keys()), linewidth=1.2, fliersize=3, ax=axes[1])
axes[1].set_title("Score Box Plot — Model × Dataset", fontsize=12, fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 11)
axes[1].axhline(7, color="gray", ls=":", lw=1, alpha=0.5, label="score=7")
axes[1].legend(title="Dataset")

plt.suptitle("LLM Judge Score Distributions (100-image validation set)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(STAGE2_DIR / "stage2_judge_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: stage2_judge_distributions.png")

In [ ]:
# Inter-model agreement table (Spearman ρ between each pair)
from itertools import combinations

print("Inter-model Spearman ρ (100-image validation set)")
print("-" * 55)
for ds in ["All", "bg60k", "ecommerce118k"]:
    sub = merged_judges if ds == "All" else merged_judges[merged_judges["dataset"] == ds]
    print(f"\n  Dataset: {ds}  (n={len(sub)})")
    for m1, m2 in combinations(list(judges.keys()), 2):
        r, p = stats.spearmanr(sub[f"score_{m1}"], sub[f"score_{m2}"])
        print(f"    {m1:<15} vs {m2:<15}  ρ = {r:.3f}  (p={p:.2e})")

print()
print("Ceiling effect summary (all images):")
for name, col in zip(judges, score_cols_j):
    s = merged_judges[col]
    pct_above_8 = (s >= 8).mean() * 100
    print(f"  {name:<15}  median={s.median():.2f}  IQR={s.quantile(0.75)-s.quantile(0.25):.2f}  % scores ≥ 8: {pct_above_8:.0f}%")

### Model Selection Decision

| Model | Median | IQR | % scores ≥ 8 | Verdict |
|---|---|---|---|---|
| GPT-4o | 9.00 | 2.00 | 74% | **Ceiling effect** — most images rated 8–10; poor discrimination on ecommerce118k |
| Qwen3-VL-8B | 8.75 | 1.75 | 71% | **Ceiling effect** — similar pattern; insensitive to text/watermark issues |
| **SEA-LION** | **7.63** | **2.56** | **54%** | **Selected** — widest discrimination, most sensitive to ecommerce-specific issues (cluttered backgrounds, promotional text overlays) |

**Why ceiling effects matter for training:** A regression model trained on labels where 74% of images score ≥ 8 learns a nearly-constant function. SEA-LION's flatter distribution gives the ViT more gradient signal across the full quality range.

**Next step:** Validate SEA-LION on Pool A (177 human-annotated images) using Spearman ρ > 0.75 gate before committing to 5,000-image labelling.

## 1. Load Pool A

In [ ]:
pool_a = pd.read_excel(SCORING_DIR / "stage1_annotation" / "pool_a_177_images.xlsx")

# Drop rows where either score is missing
df = pool_a.dropna(subset=["[SEA-LION] Overall Score", "[HUMAN] Overall Score"]).copy()
df = df.rename(columns={
    "[SEA-LION] Overall Score": "sealion",
    "[HUMAN] Overall Score":   "human",
})

print(f"Pool A rows: {len(pool_a)}  |  usable (both scores present): {len(df)}")
df[["sealion", "human"]].describe().round(2)

## 2. Inter-annotator Agreement

### 2a. Spearman Rank Correlation

In [10]:
rho, pval = stats.spearmanr(df["sealion"], df["human"])
print(f"Spearman ρ = {rho:.4f}  (p = {pval:.2e})")

Spearman ρ = 0.8879  (p = 6.82e-61)


### 2b. Cohen's Kappa (3-class binning)

In [11]:
def bin_score(s):
    """Map 1–10 score to 3 ordinal classes (aligned with scoring scripts: bins=[0,3,7,10])."""
    if s <= 3:
        return 0  # low  (1–3)
    elif s <= 7:
        return 1  # mid  (4–7)
    else:
        return 2  # high (8–10)

df["sealion_class"] = df["sealion"].apply(bin_score)
df["human_class"]   = df["human"].apply(bin_score)

kappa = cohen_kappa_score(df["human_class"], df["sealion_class"])
kappa_w = cohen_kappa_score(df["human_class"], df["sealion_class"], weights="linear")

print(f"Cohen's Kappa (unweighted): {kappa:.4f}")
print(f"Cohen's Kappa (linear weighted): {kappa_w:.4f}")

Cohen's Kappa (unweighted): 0.6491
Cohen's Kappa (linear weighted): 0.6968


### 2c. Visualisations

In [ ]:
fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# --- Scatter plot ---
ax1 = fig.add_subplot(gs[0])
ax1.scatter(df["human"], df["sealion"], alpha=0.55, edgecolors="none", s=40)
lims = [1, 10]
ax1.plot(lims, lims, "--", color="gray", lw=1, label="perfect agreement")
ax1.set_xlabel("Human Overall Score")
ax1.set_ylabel("SEA-LION Overall Score")
ax1.set_title(f"Score Scatter\nSpearman ρ = {rho:.3f}")
ax1.legend(fontsize=8)

# --- Bland–Altman ---
mean_score = (df["sealion"] + df["human"]) / 2
diff_score  = df["sealion"] - df["human"]
mean_diff   = diff_score.mean()
sd_diff     = diff_score.std()

ax2 = fig.add_subplot(gs[1])
ax2.scatter(mean_score, diff_score, alpha=0.55, edgecolors="none", s=40)
ax2.axhline(mean_diff, color="tomato",  lw=1.5, label=f"bias = {mean_diff:+.2f}")
ax2.axhline(mean_diff + 1.96*sd_diff, color="steelblue", lw=1, ls="--", label="±1.96 SD")
ax2.axhline(mean_diff - 1.96*sd_diff, color="steelblue", lw=1, ls="--")
ax2.axhline(0, color="gray", lw=0.8, ls=":")
ax2.set_xlabel("Mean of Two Raters")
ax2.set_ylabel("SEA-LION − Human")
ax2.set_title("Bland–Altman Plot")
ax2.legend(fontsize=8)

# --- Confusion matrix (3 classes) ---
ax3 = fig.add_subplot(gs[2])
labels = ["low (1–3)", "mid (4–7)", "high (8–10)"]
cm = pd.crosstab(
    pd.Categorical(df["human_class"],   categories=[0,1,2]),
    pd.Categorical(df["sealion_class"], categories=[0,1,2]),
    rownames=["Human"], colnames=["SEA-LION"],
)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels,
    ax=ax3, cbar=False,
)
ax3.set_title(f"Confusion Matrix\nκ = {kappa:.3f}  (weighted κ = {kappa_w:.3f})")

plt.suptitle("Pool A — SEA-LION vs Human Annotator Agreement", fontsize=13, y=1.02)
plt.savefig(STAGE2_DIR / "stage2_agreement_pool_a.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: stage2_agreement_pool_a.png")

### 2d. Per-dataset breakdown

In [13]:
print("Agreement breakdown by dataset:")
for ds, grp in df.groupby("Dataset"):
    r, p = stats.spearmanr(grp["sealion"], grp["human"])
    k = cohen_kappa_score(grp["human_class"], grp["sealion_class"])
    print(f"  {ds:<20}  n={len(grp):>3}  Spearman ρ={r:.3f} (p={p:.2e})  κ={k:.3f}")

Agreement breakdown by dataset:
  bg60k                 n= 93  Spearman ρ=0.646 (p=2.83e-12)  κ=0.557
  ecommerce118k         n= 84  Spearman ρ=0.925 (p=3.81e-36)  κ=0.535


### Per-Dataset Agreement Asymmetry

SEA-LION agrees strongly with humans on **ecommerce118k** (ρ ≈ 0.925) but weakly on **bg60k** (ρ ≈ 0.646). This asymmetry has two implications:

1. **Label quality**: SEA-LION's scores are noisier/more biased for bg60k than for ecommerce118k. The model trained on these labels will therefore have weaker calibration specifically on bg60k images.

2. **Pool B validation**: This explains why DINOv2 Pool B per-dataset breakdown shows a larger gain over SEA-LION on bg60k (SRCC +0.277) than on ecommerce118k. The ViT learned to correct for SEA-LION's systematic errors on that subset.

**Root cause hypothesis**: bg60k images contain richer, more ambiguous backgrounds where SEA-LION's textual reasoning diverges from human visual intuition. ecommerce118k images are more standardised (white/grey backgrounds), making the scoring rubric easier to apply consistently.

## 3. Gate Decision

In [14]:
THRESHOLD = 0.75

print("=" * 55)
print(f"  Spearman ρ = {rho:.4f}   threshold = {THRESHOLD}")
print("=" * 55)

if rho >= THRESHOLD:
    print("  ✓ PASS — SEA-LION labels are trusted as training signal.")
    print("  Proceeding to train/val split.")
else:
    print("  ✗ FAIL — Agreement too low. Do NOT use SEA-LION labels.")
    print("  Review scoring prompt or collect more human labels before continuing.")
print("=" * 55)

  Spearman ρ = 0.8879   threshold = 0.75
  ✓ PASS — SEA-LION labels are trusted as training signal.
  Proceeding to train/val split.


## 4. Train / Val Split (4,000 / 1,000)

In [ ]:
assert rho >= THRESHOLD, "Gate failed — skipping split."

finetune = pd.read_csv(STAGE2_DIR / "sealion_5000_labels.csv")
print(f"5k set shape: {finetune.shape}")
print(finetune["dataset"].value_counts())

In [ ]:
# Verify no overlap with Pool A or Pool B
pool_b = pd.read_excel(SCORING_DIR / "stage1_annotation" / "pool_b_300_images.xlsx")

ft_keys = set(zip(finetune["dataset"], finetune["filename"]))
pa_keys = set(zip(pool_a["Dataset"].str.lower(), pool_a["Filename"]))
pb_keys = set(zip(pool_b["Dataset"].str.lower(), pool_b["Filename"]))

overlap_a = len(ft_keys & pa_keys)
overlap_b = len(ft_keys & pb_keys)

print(f"5k ∩ Pool A = {overlap_a}  (expected 0)")
print(f"5k ∩ Pool B = {overlap_b}  (expected 0)")
assert overlap_a == 0 and overlap_b == 0, "Overlap detected — check sampling scripts!"

In [17]:
# Stratified split by dataset to preserve bg60k / ecommerce118k balance
SEED     = 42
VAL_FRAC = 0.20  # 1,000 / 5,000

train_parts, val_parts = [], []

for ds, grp in finetune.groupby("dataset"):
    grp = grp.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_val   = round(len(grp) * VAL_FRAC)
    n_train = len(grp) - n_val
    val_parts.append(grp.iloc[:n_val])
    train_parts.append(grp.iloc[n_val:])
    print(f"  {ds:<20}  train={n_train}  val={n_val}")

train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df   = pd.concat(val_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\nTrain: {len(train_df)}  |  Val: {len(val_df)}  |  Total: {len(train_df)+len(val_df)}")

  bg60k                 train=2051  val=513
  ecommerce118k         train=1949  val=487

Train: 4000  |  Val: 1000  |  Total: 5000


In [ ]:
# Score distribution check
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (split_name, split_df) in zip(axes, [("Train", train_df), ("Val", val_df)]):
    for ds, grp in split_df.groupby("dataset"):
        ax.hist(grp["overall_score"], bins=18, alpha=0.6, label=ds, range=(1, 10))
    ax.set_title(f"{split_name} split (n={len(split_df)})")
    ax.set_xlabel("Overall Score")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)

plt.suptitle("Score distribution — Train vs Val", fontsize=13)
plt.tight_layout()
plt.savefig(STAGE2_DIR / "stage2_train_val_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: stage2_train_val_distribution.png")

In [ ]:
# Save splits
train_out = STAGE2_DIR / "train_4000.csv"
val_out   = STAGE2_DIR / "val_1000.csv"

train_df.to_csv(train_out, index=False)
val_df.to_csv(val_out,   index=False)

print(f"Saved: {train_out}  ({len(train_df)} rows)")
print(f"Saved: {val_out}  ({len(val_df)} rows)")

## 5. Summary

In [20]:
print("=" * 55)
print("  STAGE 2 SUMMARY")
print("=" * 55)
print(f"  Pool A agreement")
print(f"    Spearman ρ       = {rho:.4f}  (threshold {THRESHOLD})")
print(f"    Cohen's κ        = {kappa:.4f}")
print(f"    Cohen's κ (wtd)  = {kappa_w:.4f}")
print(f"    Gate             = {'PASS ✓' if rho >= THRESHOLD else 'FAIL ✗'}")
print()
print(f"  Training set")
print(f"    Train            = {len(train_df):,} images")
print(f"    Val              = {len(val_df):,} images")
print(f"    Score mean (tr)  = {train_df['overall_score'].mean():.2f}")
print(f"    Score mean (val) = {val_df['overall_score'].mean():.2f}")
print("=" * 55)

  STAGE 2 SUMMARY
  Pool A agreement
    Spearman ρ       = 0.8879  (threshold 0.75)
    Cohen's κ        = 0.6491
    Cohen's κ (wtd)  = 0.6968
    Gate             = PASS ✓

  Training set
    Train            = 4,000 images
    Val              = 1,000 images
    Score mean (tr)  = 7.18
    Score mean (val) = 7.20
